# 6. Robustness

Two checks:
1. Do the results of part 4 depend on the demand rule?
2. Do the segments replicate when the same people rate the same bags a second time (the offline ratings)?

In [1]:
import sys
sys.path.append("../src")

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import adjusted_rand_score
from scipy.optimize import linear_sum_assignment

from data import load_data, encode
from models import hb
from segments import latent_class, lc_assign
from experiment import repeated_cv

design, online, offline = load_data()
X = encode(design)
cols = list(X.columns)
res = hb(online, X, seed=0)
B = res["B"].mean(0)
sd = np.sqrt(res["s2"].mean())
print("HB error sd: %.2f" % sd)

HB error sd: 0.67


## 1. Demand rule

- *threshold 3*: buy the best offered bag if its predicted rating is at least 3
- *threshold 4*: the same, but only ratings of at least 4 ("probably would buy") count as a purchase
- *smooth*: pick the bag with the highest predicted rating and buy it with probability Phi((rating - 3) / sd), where sd is the HB error standard deviation. This removes the hard cutoff at 3.

I don't use a standard logit here: in a logit every extra product raises the total purchase probability even if all consumers have the same preferences, so a product line would look better than one product for reasons unrelated to heterogeneity. With the smooth rule, offering more bags to identical consumers adds nothing.

In [2]:
demands = {
    "threshold 3": {"type": "threshold", "thr": 3},
    "threshold 4": {"type": "threshold", "thr": 4},
    "smooth": {"type": "smooth", "thr": 3, "sd": sd},
}
tables = []
for name, demand in demands.items():
    cv = pd.DataFrame(repeated_cv(online, X, B, Ks=[1, 2, 3, 4], n_reps=10, demand=demand))
    cv = cv[cv["strategy"] != "oracle"]
    mass = cv[cv["strategy"] == "mass market"].set_index("fold")[["realized", "potential"]]
    cv = cv.join(mass, on="fold", rsuffix="_mass")
    cv["realized"] = 100 * (cv["realized"] / cv["realized_mass"] - 1)
    cv["potential"] = 100 * (cv["potential"] / cv["potential_mass"] - 1)
    t = cv[cv["K"] > 1].groupby(["strategy", "method", "K"])[["realized", "potential"]].mean()
    t.columns = pd.MultiIndex.from_product([[name], t.columns])
    tables.append(t)
robust = pd.concat(tables, axis=1).round(1)
robust

threshold 3           threshold 4            \
                                    realized potential    realized potential   
strategy               method  K                                               
optimal line           -       2         2.4       6.4         5.0      10.6   
                               3         4.1      10.9         7.6      18.3   
                               4         6.6      13.5        10.0      22.0   
segment products, open LC      2         0.6       0.4         2.6       3.1   
                               3         1.1       1.7         6.3       9.5   
                               4         1.9       2.4         8.2      11.0   
                       k-means 2         2.4       1.7         3.7       2.4   
                               3        -0.2       0.8         4.9       8.1   
                               4         2.8       4.1         6.9      11.3   
targeted               LC      2        -2.0       1.0         1.4       3.7   
                               3        -3.6       4.6        -2.6       9.2   
                               4        -2.8       6.9        -2.5      13.5   
                       k-means 2        -0.5       1.9        -2.4       2.6   
                               3         0.6       5.4        -2.6       8.0   
                               4        -0.9       7.0        -2.7      12.7   

                                   smooth            
                                 realized potential  
strategy               method  K                     
optimal line           -       2      1.6       4.1  
                               3      3.6       6.8  
                               4      5.6       8.7  
segment products, open LC      2     -1.2      -0.3  
                               3     -0.3       1.5  
                               4     -0.4       2.5  
                       k-means 2     -0.4       0.3  
                               3     -0.2      -0.3  
                               4     -0.1       0.3  
targeted               LC      2     -0.3       0.7  
                               3     -0.4       2.4  
                               4     -0.3       3.5  
                       k-means 2     -0.6       0.4  
                               3      1.3       3.0  
                               4      1.0       4.4

The main conclusions hold under all three rules:
- the optimal line gains out of sample (+1.6% to +9.7%, about half of its potential);
- targeting segments (each segment sees only its own bag) realizes much less than its potential and is mostly at or below the mass market.

One difference: with threshold 4, offering the segment bags to everyone works about as well as the optimal line (+6% to +9%). With a higher threshold fewer consumers buy a single mass-market bag, so having several bags on the shelf matters more than exactly which bags they are. With the smooth rule all gains are smaller, because consumers near the cutoff now buy with a probability close to 1/2 instead of all-or-nothing.

## 2. Do segments replicate in the second measurement?

The same 122 respondents rated the same 20 bags a second time (offline). If segments describe stable consumer types, a person should end up in the same segment both times.

- *online vs offline LC*: LC estimated separately on each measurement, compare assignments (adjusted Rand index)
- *offline assigned with online model*: the online LC model assigns people using their offline ratings, i.e. how a firm would classify a returning customer
- *k-means*: k-means on HB partworths estimated separately on each measurement
- *bootstrap (part 3)*: stability under resampling within the online data, for reference

In [3]:
res_off = hb(offline, X, seed=0)
B_off = res_off["B"].mean(0)
bootstrap = {2: 0.84, 3: 0.54, 4: 0.43}

rows = []
for K in [2, 3, 4]:
    lc_on = latent_class(online, X, K)
    lc_off = latent_class(offline, X, K)
    km_on = KMeans(K, n_init=20, random_state=0).fit_predict(StandardScaler().fit_transform(B[:, 1:]))
    km_off = KMeans(K, n_init=20, random_state=0).fit_predict(StandardScaler().fit_transform(B_off[:, 1:]))
    rows.append({
        "K": K,
        "online vs offline LC": adjusted_rand_score(lc_on["labels"], lc_off["labels"]),
        "offline assigned with online model": adjusted_rand_score(lc_on["labels"], lc_assign(offline, X, lc_on).argmax(1)),
        "k-means": adjusted_rand_score(km_on, km_off),
        "bootstrap (part 3)": bootstrap[K],
    })
pd.DataFrame(rows).set_index("K").round(2)

,online vs offline LC,offline assigned with online model,k-means,bootstrap (part 3)
K,,,,
2,0.38,0.49,0.11,0.84
3,0.23,0.25,0.16,0.54
4,0.19,0.18,0.13,0.43


Segment partworths for K = 2, online and offline (offline segments matched to the online ones):

In [4]:
lc_on, lc_off = latent_class(online, X, 2), latent_class(offline, X, 2)
cost = ((lc_on["B"][:, None, :] - lc_off["B"][None, :, :]) ** 2).sum(2)
_, match = linear_sum_assignment(cost)
t = pd.DataFrame({
    "online 1": lc_on["B"][0], "offline 1": lc_off["B"][match[0]],
    "online 2": lc_on["B"][1], "offline 2": lc_off["B"][match[1]],
}, index=cols)
t.loc["share"] = [lc_on["pi"][0], lc_off["pi"][match[0]], lc_on["pi"][1], lc_off["pi"][match[1]]]
t.round(2)

,online 1,offline 1,online 2,offline 2
price,-0.85,-0.47,-0.47,-0.42
exterior_reflective,-0.25,-0.49,-0.38,-0.76
exterior_colorful,-0.25,0.07,-1.95,-1.93
exterior_blue,-0.21,-0.04,-0.23,-0.22
size_large,0.28,-0.39,0.25,-0.18
strap_pad_yes,0.70,0.37,0.31,0.07
pocket_yes,0.66,0.27,0.22,0.01
interior_file_divider,0.48,0.56,0.32,0.45
interior_laptop,0.76,0.93,0.46,0.80
share,0.52,0.61,0.48,0.39


Segments replicate poorly. With 2 segments the agreement between the two measurements is 0.38-0.49, far below the 0.84 bootstrap stability within the online data, and with 3-4 segments it drops to about 0.2. The segment that strongly dislikes the colorful exterior is found in both measurements with almost the same partworth (-1.95 vs -1.93), but the other segment changes (for example, the size preference flips sign).

Part of this is because the second measurement was done offline, with the physical bags, so preferences themselves may differ, not only measurement noise. Either way, a firm that assigns a returning customer to a segment based on a new set of ratings would often put them in a different segment. This is consistent with part 4: targeting loses its potential gain partly because new consumers are assigned to the wrong segment.